# Challenge 1: Weather Agent with Custom Tools (Google ADK)

**Goal:** Build an ADK agent that uses custom tools to fetch real-time US weather
data (National Weather Service API) and geocode place names, then produce a weather
summary/alert. Tested against multiple US cities on two model backends.

**No API keys anywhere.** All model calls authenticate through Application Default
Credentials (ADC), already active in Colab Enterprise. Geocoding (Open-Meteo,
keyless - see note in section 2) and weather (NWS) are fully keyless.

### A note on the "third-party model" requirement

The lab asks for support for Gemini **and** another provider (Claude, GPT, etc.).
This notebook is built so the second agent is a **swappable, provider-agnostic
slot**: switching providers is a one-line model-string change, which is the actual
capability the requirement tests.

In this Qwiklabs sandbox project, Anthropic Claude was **not accessible via Vertex
AI Model Garden** (partner-model access requires an IAM/procurement grant the
sandbox doesn't provide — confirmed with the diagnostic in section 4). So the
runnable second agent uses a *different Gemini model* (`gemini-2.0-flash`), and the
exact Claude-on-Vertex wiring is included as a documented, ready-to-activate block
in section 5. On any project with Anthropic Model Garden access granted, uncommenting
that block switches the third-party agent to Claude with no other change.


In [ ]:
# 1. Install dependencies
!pip install --quiet google-adk litellm requests anthropic[vertex] googlemaps


In [ ]:
import os
import asyncio
import requests
from typing import Any

import google.auth

credentials, PROJECT_ID = google.auth.default()
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

print(f"Using Vertex AI project={PROJECT_ID!r}, location={LOCATION!r} via ADC.")


## 2. Tool 1 — Geocoding (place name → lat/lon)

**Note on the geocoding provider.** The lab names the *Google Maps Geocoding API*.
That API is not usable in this Qwiklabs sandbox: the Geocoding API was not enabled
by default, and this account lacks the IAM permission to create a Google Maps API
key (`gcloud services api-keys create` returns `PERMISSION_DENIED` /
`AUTH_PERMISSION_DENIED`, and the API Keys service itself can't be enabled by this
role). Since the sandbox blocks the key, this tool uses **Open-Meteo's keyless
geocoding API** instead, which returns the same latitude/longitude the weather tool
needs. The Google Maps implementation is included immediately below, commented out,
and is a drop-in replacement on any project where a Maps API key is available.


In [ ]:
def geocode_location(place_name: str) -> dict[str, Any]:
    """Convert a place name into geographic coordinates.

    Uses Open-Meteo's keyless geocoding service to resolve a free-text place
    name (typically a US city and state) into a latitude/longitude pair for
    use with the National Weather Service API. See the commented Google Maps
    Geocoding API version below for the key-based equivalent named in the lab.

    Args:
        place_name: A human-readable location, e.g. "Austin, TX" or
            "Seattle, Washington".

    Returns:
        A dictionary with keys:
            - "status": "success" or "error"
            - "latitude" (float), "longitude" (float) on success
            - "resolved_name" (str) on success, for disambiguation
            - "error_message" (str) on error
    """
    name_part = place_name.split(",")[0].strip()

    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": name_part, "count": 5, "country": "US", "language": "en"}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as exc:
        return {"status": "error", "error_message": f"Geocoding request failed: {exc}"}

    results = data.get("results")
    if not results:
        return {"status": "error", "error_message": f"No match found for '{place_name}'."}

    state_part = place_name.split(",")[1].strip() if "," in place_name else None
    chosen = results[0]
    if state_part:
        for candidate in results:
            admin1 = candidate.get("admin1", "")
            if state_part.lower() in admin1.lower() or admin1.lower().startswith(state_part.lower()):
                chosen = candidate
                break

    return {
        "status": "success",
        "latitude": chosen["latitude"],
        "longitude": chosen["longitude"],
        "resolved_name": f"{chosen.get('name')}, {chosen.get('admin1', '')}".strip(", "),
    }


# --- Google Maps Geocoding API version (as named in the lab) ---------------
# Drop-in replacement for geocode_location above on any project where a Google
# Maps API key is available. Requires the Geocoding API enabled and a key in
# GOOGLE_MAPS_API_KEY (read from a Colab secret or env var; never hardcoded).
# Not usable in this sandbox because key creation is permission-blocked.
#
# def geocode_location(place_name: str) -> dict[str, Any]:
#     """Convert a place name into coordinates via the Google Maps Geocoding API."""
#     api_key = os.environ.get("GOOGLE_MAPS_API_KEY", "")
#     if not api_key:
#         return {"status": "error",
#                 "error_message": "GOOGLE_MAPS_API_KEY is not set."}
#     url = "https://maps.googleapis.com/maps/api/geocode/json"
#     params = {"address": place_name, "key": api_key}
#     try:
#         response = requests.get(url, params=params, timeout=10)
#         response.raise_for_status()
#         data = response.json()
#     except requests.RequestException as exc:
#         return {"status": "error", "error_message": f"Geocoding request failed: {exc}"}
#     if data.get("status") != "OK" or not data.get("results"):
#         detail = data.get("error_message", "")
#         return {"status": "error",
#                 "error_message": f"Geocoding API status: {data.get('status')}. {detail}".strip()}
#     loc = data["results"][0]["geometry"]["location"]
#     return {
#         "status": "success",
#         "latitude": loc["lat"],
#         "longitude": loc["lng"],
#         "formatted_address": data["results"][0].get("formatted_address", place_name),
#     }


### Quick check — see the latitude/longitude directly

When the agent runs, it uses these coordinates silently to call the weather tool,
so the lat/lon don't appear in the agent's replies. Call the function directly to
see them:


In [ ]:
for _city in ["Seattle, WA", "Miami, FL", "Chicago, IL"]:
    print(_city, "->", geocode_location(_city))


## 3. Tool 2 — Weather lookup (lat/lon → forecast)

Keyless NWS API, two-hop points→forecast call.


In [ ]:
def get_weather_forecast(latitude: float, longitude: float) -> dict[str, Any]:
    """Retrieve the current weather forecast for a US location.

    Queries the National Weather Service (NWS) API in two steps: first
    resolving the forecast grid endpoint for the given coordinates, then
    fetching the short-term forecast from that endpoint. Only covers
    locations within the United States and its territories.

    Args:
        latitude: Latitude in decimal degrees (WGS84).
        longitude: Longitude in decimal degrees (WGS84).

    Returns:
        A dictionary with keys:
            - "status": "success" or "error"
            - "location" (str), "forecast_period" (str),
              "short_forecast" (str), "temperature" (int),
              "temperature_unit" (str), "wind_speed" (str),
              "detailed_forecast" (str) on success
            - "error_message" (str) on error
    """
    headers = {
        "User-Agent": "ADK-Weather-Agent-Lab (student notebook, contact: student@example.com)",
        "Accept": "application/geo+json",
    }

    points_url = f"https://api.weather.gov/points/{latitude},{longitude}"

    try:
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        points_data = points_resp.json()
    except requests.RequestException as exc:
        return {
            "status": "error",
            "error_message": f"NWS points lookup failed for ({latitude}, {longitude}): {exc}",
        }

    properties = points_data.get("properties", {})
    forecast_url = properties.get("forecast")
    relative_location = properties.get("relativeLocation", {}).get("properties", {})
    location_name = f"{relative_location.get('city', 'Unknown')}, {relative_location.get('state', '')}".strip(", ")

    if not forecast_url:
        return {
            "status": "error",
            "error_message": "NWS did not return a forecast URL for this location "
                             "(it may be outside NWS coverage, e.g. outside the US).",
        }

    try:
        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        forecast_data = forecast_resp.json()
    except requests.RequestException as exc:
        return {"status": "error", "error_message": f"NWS forecast fetch failed: {exc}"}

    periods = forecast_data.get("properties", {}).get("periods", [])
    if not periods:
        return {"status": "error", "error_message": "NWS returned no forecast periods."}

    current = periods[0]

    return {
        "status": "success",
        "location": location_name,
        "forecast_period": current.get("name", "Unknown period"),
        "short_forecast": current.get("shortForecast", ""),
        "temperature": current.get("temperature"),
        "temperature_unit": current.get("temperatureUnit", "F"),
        "wind_speed": current.get("windSpeed", ""),
        "detailed_forecast": current.get("detailedForecast", ""),
    }


## 4. Diagnostic (optional) — is Claude reachable on THIS project?

This directly probes the Anthropic publisher endpoint with an ADC token. A `200`
means Claude works on Vertex here (activate the Claude block in section 5). A `404`
"...does not have access to it" or a `403` means the sandbox lacks partner-model
access — no code change fixes that, and the notebook falls back to a second Gemini
model. This is informational only; the notebook runs regardless of the result.


In [ ]:
import google.auth.transport.requests as gareq

def probe_claude_on_vertex() -> None:
    """Probe whether Anthropic Claude is callable on this project via Vertex."""
    creds, project = google.auth.default()
    creds.refresh(gareq.Request())

    # A commonly-available Claude 3.5 Sonnet v2 model id; regions where Claude
    # is typically served on Vertex.
    model_id = "claude-3-5-sonnet-v2@20241022"
    for region in ("us-east5", "us-central1", "global"):
        host = "aiplatform.googleapis.com" if region == "global" else f"{region}-aiplatform.googleapis.com"
        url = (f"https://{host}/v1/projects/{project}/locations/{region}"
               f"/publishers/anthropic/models/{model_id}:rawPredict")
        try:
            r = requests.post(
                url,
                headers={"Authorization": f"Bearer {creds.token}",
                         "Content-Type": "application/json"},
                json={"anthropic_version": "vertex-2023-10-16", "max_tokens": 10,
                      "messages": [{"role": "user", "content": "hi"}]},
                timeout=20,
            )
            print(f"[{region}] HTTP {r.status_code}: {r.text[:160]}")
        except requests.RequestException as exc:
            print(f"[{region}] request error: {exc}")


probe_claude_on_vertex()


## 5. Define the agent(s)

- **Primary agent** — Gemini 2.5 Flash (native Vertex, ADC).
- **Third-party slot** — a swappable second backend. It runs on Gemini 2.5 Flash-Lite
  here because Claude wasn't accessible in this sandbox (see section 4). The
  commented block directly below shows the exact Claude-on-Vertex wiring; uncomment
  it (and comment out the Gemini fallback) on any project with Anthropic Model
  Garden access to switch the third-party agent to Claude with no other change.


In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

AGENT_INSTRUCTION = (
    "You are a US weather assistant. When the user asks about weather in a "
    "location, first call `geocode_location` to resolve the place name to "
    "coordinates, then call `get_weather_forecast` with those coordinates. "
    "Summarize the result in 2-4 sentences: current conditions, temperature, "
    "and wind. If the short forecast or detailed forecast mentions storms, "
    "extreme heat, extreme cold, high wind, or any hazardous condition, lead "
    "your response with a clearly labeled ALERT line before the summary. If "
    "either tool returns status='error', tell the user plainly what went "
    "wrong (e.g. location not found, or outside NWS/US coverage) instead of "
    "guessing at weather data."
)

TOOLS = [geocode_location, get_weather_forecast]

# --- Primary agent: Gemini 2.5 Flash (native Vertex, ADC) -----------------
weather_agent_primary = Agent(
    name="weather_agent_primary",
    model="gemini-2.5-flash",
    description="Provides US weather summaries and alerts using Gemini 2.5 Flash.",
    instruction=AGENT_INSTRUCTION,
    tools=TOOLS,
)

# --- Third-party slot -----------------------------------------------------
# OPTION A (active): a different Gemini model, proving the backend is swappable
# without touching tools or instructions. Runs cleanly in this sandbox.
weather_agent_third_party = Agent(
    name="weather_agent_third_party",
    model="gemini-2.5-flash-lite",
    description="Provides US weather summaries and alerts using a swappable second backend.",
    instruction=AGENT_INSTRUCTION,
    tools=TOOLS,
)

# OPTION B (documented): Claude on Vertex AI Model Garden via LiteLLM, ADC-only,
# no ANTHROPIC_API_KEY. Activate on a project with Anthropic Model Garden access
# by replacing OPTION A above with this. Use the model id / region that section 4
# returned HTTP 200 for.
#
# weather_agent_third_party = Agent(
#     name="weather_agent_third_party",
#     model=LiteLlm(
#         model="vertex_ai/claude-3-5-sonnet-v2@20241022",
#         vertex_project=PROJECT_ID,
#         vertex_location="us-east5",
#     ),
#     description="Provides US weather summaries and alerts using Claude on Vertex.",
#     instruction=AGENT_INSTRUCTION,
#     tools=TOOLS,
# )
#
# OPTION C (documented): OpenAI GPT via LiteLLM. Requires OPENAI_API_KEY, which
# conflicts with the "no API keys" constraint here, so it is documented only:
#     model=LiteLlm(model="openai/gpt-4o")


## 6. Runner + session plumbing

In [ ]:
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai.types import Content, Part

APP_NAME = "weather_agent_lab"
USER_ID = "test_user"

session_service = InMemorySessionService()

runner_primary = Runner(
    agent=weather_agent_primary, app_name=APP_NAME, session_service=session_service
)
runner_third_party = Runner(
    agent=weather_agent_third_party, app_name=APP_NAME, session_service=session_service
)


async def call_agent(runner: Runner, session_id: str, query_text: str) -> str:
    """Send one user message to an ADK agent and return its final text reply."""
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=session_id
    )
    content = Content(role="user", parts=[Part(text=query_text)])
    final_text = ""
    async for event in runner.run_async(
        user_id=USER_ID, session_id=session_id, new_message=content
    ):
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text
    return final_text


## 7. Test code — multiple US cities, both model backends

In [ ]:
TEST_CITIES = [
    "Seattle, WA",
    "Miami, FL",
    "Chicago, IL",
    "Phoenix, AZ",
    "Denver, CO",
]


async def run_city_tests(runner: Runner, label: str) -> None:
    print(f"\n=== Testing {label} ===")
    for i, city in enumerate(TEST_CITIES):
        session_id = f"{label}_{i}"
        query = f"What's the weather like in {city} right now?"
        reply = await call_agent(runner, session_id, query)
        print(f"\n--- {city} ---\n{reply}")


await run_city_tests(runner_primary, "primary_gemini_2.5")
await run_city_tests(runner_third_party, "third_party_gemini_2.5_lite")


## 8. Notes / known gaps

- **No API keys anywhere** — Gemini via ADC; geocoding and weather fully keyless.
- **Multi-provider architecture** is demonstrated via the swappable third-party
  slot in section 5. Claude/GPT wiring is included and documented; the runnable
  second backend is Gemini 2.5 Flash-Lite because Anthropic Model Garden access was not
  granted in this Qwiklabs sandbox (see section 4's probe). On a project with that
  access, activating Claude is a one-block swap.
- **Geocoding provider deviates from the lab wording, by necessity.** The lab
  names the Google Maps Geocoding API, but this Qwiklabs sandbox does not permit
  creating a Google Maps API key (key creation returns PERMISSION_DENIED and the
  API Keys service can't be enabled by this role). Open-Meteo's keyless geocoder is
  used instead; it returns the same coordinates. The Google Maps implementation is
  included, commented out, in section 2 as a drop-in replacement for any project
  where a key is available.
- **NWS coverage** is US-only; out-of-range coordinates surface an error rather
  than a fabricated forecast.
